# FRInGe and FRInGe-B: end-to-end sanity check

This notebook runs the **current canonical implementation** used by the benchmark on one included ImageNet image. It compares:

- **FRInGe**: full categorical Fisher–Rao geometry;
- **FRInGe-B**: the target-versus-rest binary simplification.

Both methods use the model's predicted top-1 class as the target and load the released ResNet-18 hyperparameters from `configs/ablation/`. The notebook reports numerical sanity checks, completeness residuals, FRInGe-B endpoint diagnostics, and attribution visualizations.

> `RUN_MODE = "quick"` reduces the waypoint and solver budgets of categorical FRInGe for an interactive smoke test. FRInGe-B is already fast and retains its released settings so that its endpoint diagnostics remain meaningful. Quick-mode FRInGe is **not** a reproduction of the paper result; set `RUN_MODE = "paper"` to use both released parameter sets unchanged.


### Why the binary variant is cheaper

FRInGe-B collapses the categorical prediction to `[p_t, 1 - p_t]`. If `u` is the input gradient of the target-versus-rest log-odds, its pullback Fisher matrix is

$$G_{\mathrm{B}} = p_t(1-p_t)\,u u^\top,$$

which is rank one. With isotropic damping, the natural-gradient direction therefore has a closed form instead of requiring the full categorical matrix-free conjugate-gradient solve. The released smoothed configuration used here adds a spatial operator and applies a structured Sherman–Morrison solve around that regularizer; it should not be described as the unsmoothed closed-form case.


## 1. Setup

Run Jupyter from the repository root. The fallback below also handles a kernel whose working directory is `notebooks/`. Pretrained torchvision weights are downloaded on first use.


In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf
from torchvision.models import ResNet18_Weights

cwd = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (cwd, cwd.parent)
     if (candidate / "methods" / "FisherRaoIG").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start this notebook from the FRInGe repository root or notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common.utils import denormalize, load_image, load_model
from methods.FisherRaoIG.FisherRaoIntegratedGradients import (
    FisherRaoIntegratedGradients,
)

MODEL_NAME = "resnet18"
IMAGE_PATH = ROOT / "examples" / "n01580077_jay.JPEG"
RUN_MODE = "quick"  # "quick" or "paper"
QUICK_KL_TARGET = 2.0e-2
QUICK_SOLVER_ITERS = 5
SEED = 7

if RUN_MODE not in {"quick", "paper"}:
    raise ValueError("RUN_MODE must be 'quick' or 'paper'.")
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repository: {ROOT}")
print(f"Device: {device} | mode: {RUN_MODE}")


## 2. Load the model and define the target

The benchmark explains each model's predicted class rather than using the dataset label. This avoids silently changing the attribution target for misclassified images.


In [ ]:
model, preprocess = load_model(MODEL_NAME, device=device)
image = load_image(IMAGE_PATH, preprocess, device=device)

with torch.no_grad():
    logits = model(image)
    probabilities = logits.softmax(dim=1)
    target_idx = probabilities.argmax(dim=1)
    target_probability = probabilities.gather(1, target_idx[:, None]).item()

categories = ResNet18_Weights.IMAGENET1K_V1.meta["categories"]
target_label = categories[target_idx.item()]
print(
    f"Target: class {target_idx.item()} ({target_label}); "
    f"confidence={target_probability:.4f}"
)

plt.figure(figsize=(4.5, 4.5))
plt.imshow(denormalize(image[0]))
plt.title(f"{target_label} — p={target_probability:.3f}")
plt.axis("off")
plt.tight_layout()
plt.show()


## 3. Load the released method configurations

Paper mode leaves both files unchanged. Quick mode increases the per-waypoint KL budget and truncates the iterative inverse solve only for categorical FRInGe. FRInGe-B keeps its released parameters because its binary structure is already inexpensive and aggressive truncation materially degrades its endpoint convergence.


In [ ]:
def load_ablation_parameters(filename: str, model_name: str) -> dict:
    config = OmegaConf.to_container(
        OmegaConf.load(ROOT / "configs" / "ablation" / filename),
        resolve=True,
    )
    model_parameters = config.get(model_name)
    if not isinstance(model_parameters, dict):
        raise KeyError(f"No parameters for {model_name!r} in {filename}.")
    shared = {
        key: value
        for key, value in config.items()
        if key != "name" and not isinstance(value, dict)
    }
    return {**shared, **model_parameters}


def parameters_for_run(parameters: dict) -> dict:
    parameters = dict(parameters)
    if RUN_MODE == "quick" and not bool(parameters.get("binary", False)):
        parameters["kl_target"] = max(
            float(parameters["kl_target"]), QUICK_KL_TARGET
        )
        solver_key = (
            "cg_max_iters" if "cg_max_iters" in parameters else "Ainv_iters"
        )
        parameters[solver_key] = min(
            int(parameters[solver_key]), QUICK_SOLVER_ITERS
        )
    return parameters


fringe_parameters = parameters_for_run(
    load_ablation_parameters("3_fisher_smooth.yaml", MODEL_NAME)
)
fringeb_parameters = parameters_for_run(
    load_ablation_parameters("fringe2_binary.yaml", MODEL_NAME)
)

settings_table = pd.DataFrame(
    [
        {
            "method": name,
            "binary": bool(params.get("binary", False)),
            "kl_target": float(params["kl_target"]),
            "eta_max": float(params["eta_max"]),
            "delta_euc": float(params["delta_euc"]),
            "regularization": float(
                params.get("lambda_ratio", params.get("lam"))
            ),
            "solver_iters": int(
                params.get("cg_max_iters", params.get("Ainv_iters"))
            ),
        }
        for name, params in (
            ("FRInGe", fringe_parameters),
            ("FRInGe-B", fringeb_parameters),
        )
    ]
)
settings_table


## 4. Run FRInGe and FRInGe-B

Both calls go through `FisherRaoIntegratedGradients.attribute`, exactly as in the benchmark runner. The `binary` flag selects the target-versus-rest backend. Do not wrap attribution calls in `torch.no_grad()`: both methods require model derivatives.


In [ ]:
def target_score(inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return model(inputs).gather(1, targets[:, None]).squeeze(1)


def run_fringe(name: str, parameters: dict) -> dict:
    solver_iters = int(
        parameters.get("cg_max_iters", parameters.get("Ainv_iters", 20))
    )
    explainer = FisherRaoIntegratedGradients(
        model=model,
        model_forward=target_score,
        target_idx=target_idx,
        cg_max_iters=solver_iters,
    )

    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    attributions, completeness_delta = explainer.attribute(
        x=image,
        kl_target=float(parameters["kl_target"]),
        fisher=bool(parameters.get("fisher", True)),
        binary=bool(parameters.get("binary", False)),
        delta_euc=float(parameters["delta_euc"]),
        eta_max=float(parameters["eta_max"]),
        use_sobolev_preconditioner=bool(
            parameters.get("use_sobolev_preconditioner", True)
        ),
        lambda_ratio=float(
            parameters.get("lambda_ratio", parameters.get("lam"))
        ),
        smoothing=bool(parameters.get("smoothing", False)),
        gamma_step=float(parameters.get("gamma_step", 0.0)),
        gamma_prior=float(parameters.get("gamma_prior", 0.0)),
    )
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    binary_diagnostics = None
    stats = getattr(explainer, "last_binary_stats", None)
    if stats is not None:
        binary_diagnostics = {
            "steps": int(stats["total_steps"]),
            "relative_completeness_error": float(
                stats["completeness_relative_error"][0]
            ),
            "endpoint_target_probability": float(
                stats["endpoint_target_probability"][0]
            ),
            "endpoint_probability_error": float(
                stats["endpoint_error_probability"][0]
            ),
            "endpoint_binary_kl": float(stats["endpoint_error_kl"][0]),
            "endpoint_full_kl_to_uniform": float(
                stats["endpoint_full_kl_to_uniform"][0]
            ),
            "active_constraints": dict(stats["constraint_active"]),
        }

    model.zero_grad(set_to_none=True)
    result = {
        "name": name,
        "attributions": attributions.detach().cpu(),
        "completeness_delta": float(completeness_delta),
        "elapsed_seconds": elapsed,
        "binary_diagnostics": binary_diagnostics,
    }
    print(
        f"{name}: {elapsed:.2f} s, "
        f"absolute completeness residual={float(completeness_delta):.6g}"
    )
    return result


In [ ]:
fringe_result = run_fringe("FRInGe", fringe_parameters)


In [ ]:
fringeb_result = run_fringe("FRInGe-B", fringeb_parameters)


## 5. Numerical diagnostics

The **completeness residual** checks the numerical path integral against the target-logit difference between the realized endpoints. It is not an accuracy score and it does not, by itself, prove that the intended maximum-entropy endpoint was reached.

For FRInGe-B, endpoint convergence is evaluated in the collapsed binary distribution: the target probability should approach `1 / C`. The full categorical KL may remain nonzero because FRInGe-B constrains only **target versus rest**, not how probability is distributed among the non-target classes.


In [ ]:
results = [fringe_result, fringeb_result]
summary_rows = []
for result in results:
    attr = result["attributions"]
    diagnostics = result["binary_diagnostics"] or {}
    summary_rows.append(
        {
            "method": result["name"],
            "runtime_s": result["elapsed_seconds"],
            "absolute_completeness_residual": result["completeness_delta"],
            "relative_completeness_error": diagnostics.get(
                "relative_completeness_error", np.nan
            ),
            "attribution_l1": float(attr.abs().sum()),
            "all_values_finite": bool(torch.isfinite(attr).all()),
            "binary_endpoint_probability_error": diagnostics.get(
                "endpoint_probability_error", np.nan
            ),
            "binary_endpoint_kl": diagnostics.get("endpoint_binary_kl", np.nan),
        }
    )

summary = pd.DataFrame(summary_rows)
summary


In [ ]:
pd.Series(fringeb_result["binary_diagnostics"], name="FRInGe-B diagnostic")


## 6. Compare the attribution maps

Signed maps sum over input channels. Magnitude maps sum absolute channel contributions. A common color scale is used across the two methods so that visual intensity remains comparable.


In [ ]:
def attribution_maps(attributions: torch.Tensor) -> tuple[np.ndarray, np.ndarray]:
    attribution = attributions.squeeze(0)
    signed = attribution.sum(dim=0).numpy()
    magnitude = attribution.abs().sum(dim=0).numpy()
    return signed, magnitude


plot_data = [
    (result["name"], *attribution_maps(result["attributions"]))
    for result in results
]
signed_limit = max(
    np.percentile(np.abs(signed), 99.5) for _, signed, _ in plot_data
)
magnitude_limit = max(
    np.percentile(magnitude, 99.5) for _, _, magnitude in plot_data
)
signed_limit = max(float(signed_limit), 1e-12)
magnitude_limit = max(float(magnitude_limit), 1e-12)
input_image = denormalize(image[0])

fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)
for row, (name, signed, magnitude) in enumerate(plot_data):
    signed_artist = axes[row, 0].imshow(
        signed, cmap="RdBu_r", vmin=-signed_limit, vmax=signed_limit
    )
    axes[row, 0].set_title(f"{name}: signed")

    magnitude_artist = axes[row, 1].imshow(
        magnitude, cmap="magma", vmin=0.0, vmax=magnitude_limit
    )
    axes[row, 1].set_title(f"{name}: magnitude")

    axes[row, 2].imshow(input_image)
    overlay_alpha = 0.8 * np.clip(magnitude / magnitude_limit, 0.0, 1.0)
    axes[row, 2].imshow(
        magnitude, cmap="inferno", vmin=0.0, vmax=magnitude_limit, alpha=overlay_alpha
    )
    axes[row, 2].set_title(f"{name}: overlay")

    for axis in axes[row]:
        axis.axis("off")

fig.colorbar(signed_artist, ax=axes[:, 0], shrink=0.72, label="signed attribution")
fig.colorbar(
    magnitude_artist, ax=axes[:, 1:], shrink=0.72, label="attribution magnitude"
)
plt.show()


## 7. Minimal validity checks

These assertions detect broken execution, shape errors, NaNs, and infinities. They deliberately do not impose an arbitrary completeness threshold: the residual depends on path discretization and solver truncation, especially in quick mode.


In [ ]:
for result in results:
    attribution = result["attributions"]
    assert attribution.shape == image.cpu().shape
    assert torch.isfinite(attribution).all()
    assert np.isfinite(result["completeness_delta"])
    assert float(attribution.abs().sum()) > 0.0
print("FRInGe and FRInGe-B both passed the finite-value and shape checks.")


## Interpretation and next step

- Use `RUN_MODE = "quick"` only to verify installation and inspect the API; in this mode only categorical FRInGe is approximated more aggressively.
- Use `RUN_MODE = "paper"` for the released ResNet-18 settings; expect substantially longer execution, particularly for categorical FRInGe.
- Do not interpret the two wall-clock times above as a controlled speed benchmark. A valid comparison requires matched hardware, warm-up, synchronization, repeated measurements, and an explicitly matched accuracy or solver budget.
- For full-dataset evaluation, use `python -m methods.FisherRaoIG.FisherRaoIG_benchmarking` and select `ablation=fringe2_binary` for FRInGe-B.
